# Решение на TCN (Temporal Convolutional Network)

Архитектура из статьи **Bai, Kolter, Koltun (2018) «An Empirical Evaluation of
Generic Convolutional and Recurrent Networks for Sequence Modeling»**
(arXiv:1803.01271, файл `../1803.01271v2.pdf`).

Ключевые элементы TCN (по статье):
* **каузальные дилатированные свёртки**: выход в момент t зависит только от
  t и более ранних отсчётов; дилатация растёт как d = 2^i по уровням, что даёт
  экспоненциально растущее рецептивное поле при небольшой глубине;
* **residual-блок**: два слоя [дилатированная свёртка → WeightNorm → ReLU →
  spatial Dropout] + остаточная связь (1×1 свёртка, если число каналов меняется);
* признак последовательности — **выход последнего таймстепа** (он «видит» всю
  последовательность).

Наша адаптация: вход — профиль поглощения (2 канала: линейный + лог), 4 уровня
с каналами (64, 64, 128, 128), ядро k=5, дилатации 1–8 → рецептивное поле
1 + 2·(k−1)·(1+2+4+8) = 121 > 101 точки, т.е. последний таймстеп покрывает весь
профиль. Поверх энкодера — тот же протокол, что в `dl_pro.ipynb`
(WTA-мультиголова × 5 сетей + ранкер + отбор 5 кандидатов), поэтому результат
напрямую сравним с CNN-энкодером. Без синтетических данных.

In [1]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from scipy.spatial.distance import cdist

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEVICE)

device: mps


## 1. Данные

Пайплайн идентичен `dl_pro.ipynb`: 2-канальный вход (линейный + лог-профиль), сплит 80/20 (`random_state=42`, стратификация по `H2a`), PCA-16 лог-профилей для ранкера.

In [2]:
DATA_DIR = "new_dataset_V3"
GRID = np.arange(-5.0, 5.0 + 1e-9, 0.1)

profiles, targets = [], []
for name in sorted(os.listdir(DATA_DIR)):
    path = os.path.join(DATA_DIR, name)
    if not os.path.isdir(path):
        continue
    params = {}
    with open(os.path.join(path, "parameters.txt")) as f:
        for line in f:
            parts = line.split()
            if len(parts) >= 2:
                params[parts[0]] = parts[1]
    arr = np.loadtxt(os.path.join(path, "Absorption.dat"), skiprows=1)
    if arr[-1, 0] - arr[0, 0] < 5:
        continue
    profiles.append(np.interp(GRID, arr[:, 0], arr[:, 1], left=0.0, right=0.0))
    targets.append([float(params["XUVInt"]), float(params["Helium"]),
                    float(params["Msw"]), int(params["H2a"])])

profiles = np.array(profiles, dtype=np.float32)
targets = np.array(targets)
y_reg = np.log10(targets[:, :3])
y_cls = targets[:, 3].astype(np.float32)
idx_train, idx_val = train_test_split(
    np.arange(len(profiles)), test_size=0.2, random_state=SEED, stratify=y_cls)
y_mean, y_std = y_reg[idx_train].mean(0), y_reg[idx_train].std(0)
x_scale = profiles[idx_train].max()

log_prof = np.log10(np.clip(profiles, 1e-7, None))
mu = log_prof[idx_train].mean(0)
_, _, Vt = np.linalg.svd(log_prof[idx_train] - mu, full_matrices=False)
Z = (log_prof - mu) @ Vt[:16].T
Z = ((Z - Z[idx_train].mean(0)) / Z[idx_train].std(0)).astype(np.float32)


def make_input(profile):
    linear = profile / x_scale
    log = np.log10(np.clip(profile, 1e-7, None))
    log = (log - log.mean()) / (log.std() + 1e-9)
    return np.stack([linear, log]).astype(np.float32)


class ProfileDataset(Dataset):
    def __init__(self, indices, augment=False):
        self.indices, self.augment = indices, augment

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        profile = profiles[idx]
        if self.augment:
            profile = np.clip(profile + np.random.normal(
                0, 0.005 * profile.max(), profile.shape), 1e-7, None)
        return (torch.tensor(make_input(profile)),
                torch.tensor((y_reg[idx] - y_mean) / y_std, dtype=torch.float32),
                torch.tensor(y_cls[idx]))


val_loader = DataLoader(ProfileDataset(idx_val), batch_size=256)
print("train:", len(idx_train), " val:", len(idx_val))

train: 490  val: 123


## 2. Энкодер из статьи

In [3]:
from torch.nn.utils import weight_norm


class Chomp(nn.Module):
    """Обрезает правый паддинг — сохраняет каузальность (см. рис. 1 статьи)."""
    def __init__(self, n):
        super().__init__()
        self.n = n

    def forward(self, x):
        return x[:, :, :-self.n] if self.n > 0 else x


class TCNBlock(nn.Module):
    """Residual-блок TCN: [WN-свёртка -> ReLU -> Dropout] x2 + shortcut."""
    def __init__(self, cin, cout, k, d, dropout=0.2):
        super().__init__()
        pad = (k - 1) * d
        self.net = nn.Sequential(
            weight_norm(nn.Conv1d(cin, cout, k, padding=pad, dilation=d)),
            Chomp(pad), nn.ReLU(), nn.Dropout1d(dropout),
            weight_norm(nn.Conv1d(cout, cout, k, padding=pad, dilation=d)),
            Chomp(pad), nn.ReLU(), nn.Dropout1d(dropout))
        self.down = nn.Conv1d(cin, cout, 1) if cin != cout else None
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.down is None else self.down(x)
        return self.relu(out + res)


class TCNEncoder(nn.Module):
    def __init__(self, cin=2, channels=(64, 64, 128, 128), k=5):
        super().__init__()
        layers = []
        for i, c in enumerate(channels):
            layers.append(TCNBlock(cin if i == 0 else channels[i - 1], c, k, 2 ** i))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)[:, :, -1]          # последний таймстеп


FEAT_DIM = 128
def make_encoder():
    return TCNEncoder()

n = sum(p.numel() for p in TCNEncoder().parameters())
print(f"TCN-энкодер: {n/1e3:.0f}K параметров, рецептивное поле "
      f"{1 + 2*(5-1)*(1+2+4+8)} точек (профиль: 101)")

TCN-энкодер: 359K параметров, рецептивное поле 121 точек (профиль: 101)


/opt/anaconda3/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


## 3. Обучение: WTA-мультиголова × 5 сетей

Протокол тот же, что в `dl_pro.ipynb`: 5 голов, winner-takes-all лосс, 400 эпох, Adam + cosine, аугментация шумом, чекпоинт по val loss.

In [4]:
bce = nn.BCEWithLogitsLoss()
true_log = y_reg[idx_val]
true_lin = 10 ** true_log
N_HEADS = 5


class MultiHeadNet(nn.Module):
    """Энкодер из статьи + 5 WTA-голов регрессии + голова H2a."""
    def __init__(self, encoder, feat_dim):
        super().__init__()
        self.encoder = encoder
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Linear(feat_dim, 256), nn.ReLU(), nn.Dropout(0.2),
                          nn.Linear(256, 3)) for _ in range(N_HEADS)])
        self.head_cls = nn.Sequential(
            nn.Linear(feat_dim, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 1))

    def forward(self, x):
        h = self.encoder(x)
        reg = torch.stack([head(h) for head in self.heads], 1)
        return reg, self.head_cls(h).squeeze(-1)


def train_model(seed, epochs=400):
    torch.manual_seed(seed)
    np.random.seed(seed)
    tl = DataLoader(ProfileDataset(idx_train, augment=True),
                    batch_size=64, shuffle=True)
    m = MultiHeadNet(make_encoder(), FEAT_DIM).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best, best_state = float("inf"), None
    for ep in range(epochs):
        m.train()
        for x, y, c in tl:
            x, y, c = x.to(DEVICE), y.to(DEVICE), c.to(DEVICE)
            reg, cls = m(x)
            per = ((reg - y.unsqueeze(1)) ** 2).mean(-1)
            loss = per.min(1).values.mean() + 0.1 * per.mean() + bce(cls, c)
            opt.zero_grad()
            loss.backward()
            opt.step()
        sched.step()
        m.eval()
        v = 0.0
        with torch.no_grad():
            for x, y, c in val_loader:
                x, y, c = x.to(DEVICE), y.to(DEVICE), c.to(DEVICE)
                reg, cls = m(x)
                per = ((reg - y.unsqueeze(1)) ** 2).mean(-1)
                v += (per.min(1).values.mean() + 0.1 * per.mean()
                      + bce(cls, c)).item() * len(x)
        v /= len(idx_val)
        if v < best:
            best = v
            best_state = {k: t.cpu().clone() for k, t in m.state_dict().items()}
    m.load_state_dict(best_state)
    m.eval()
    PR, PC = [], []
    with torch.no_grad():
        for x, _, _ in val_loader:
            reg, cls = m(x.to(DEVICE))
            PR.append(reg.cpu().numpy())
            PC.append(torch.sigmoid(cls).cpu().numpy())
    return np.concatenate(PR) * y_std + y_mean, np.concatenate(PC)


def oracle(cands, j, log_scale=False):
    if log_scale:
        return 100 * np.mean(np.abs(cands[:, :, j] - true_log[:, j:j+1]).min(1)
                             / np.abs(true_log[:, j]))
    return 100 * np.mean(np.abs(10**cands[:, :, j] - true_lin[:, j:j+1]).min(1)
                         / true_lin[:, j])


val_cands, val_proba = [], []
for seed in range(5):
    vp, vc = train_model(seed)
    val_cands.append(vp)
    val_proba.append(vc)
    print(f"модель {seed}: oracle-of-5 XUV {oracle(vp, 0):.1f}%  "
          f"He {oracle(vp, 1):.1f}%  AUC {roc_auc_score(y_cls[idx_val], vc):.3f}")
val_cands = np.concatenate([v[:, None, :, :] for v in val_cands], 1)
val_cands = val_cands.reshape(len(idx_val), -1, 3)
val_proba = np.mean(val_proba, 0)
print(f"\noracle-of-25: XUV {oracle(val_cands, 0):.1f}%  "
      f"He {oracle(val_cands, 1):.1f}%  logMsw {oracle(val_cands, 2, True):.1f}%")

модель 0: oracle-of-5 XUV 19.9%  He 14.5%  AUC 0.976


/opt/anaconda3/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


модель 1: oracle-of-5 XUV 18.0%  He 11.9%  AUC 0.968


/opt/anaconda3/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


модель 2: oracle-of-5 XUV 20.4%  He 13.4%  AUC 0.977


/opt/anaconda3/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


модель 3: oracle-of-5 XUV 21.8%  He 16.0%  AUC 0.971


/opt/anaconda3/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


модель 4: oracle-of-5 XUV 20.3%  He 15.6%  AUC 0.980

oracle-of-25: XUV 11.0%  He 5.6%  logMsw 0.4%


## 4. Ранкер + отбор 5 кандидатов и итоговая таблица

In [5]:
torch.manual_seed(0)
np.random.seed(0)
yn = (y_reg - y_mean) / y_std
Ztr, yn_tr = Z[idx_train], yn[idx_train]
D_prof, D_par = cdist(Ztr, Ztr), cdist(yn_tr, yn_tr)
pair_x, pair_y = [], []
for a in range(len(idx_train)):
    pair_x.append(np.concatenate([Ztr[a], yn_tr[a]])); pair_y.append(1.0)
    for _ in range(2):
        pair_x.append(np.concatenate([Ztr[a], yn_tr[a] + np.random.normal(0, 0.05, 3)]))
        pair_y.append(1.0)
    for _ in range(3):
        b = np.random.randint(len(idx_train))
        if D_par[a, b] > 0.5:
            pair_x.append(np.concatenate([Ztr[a], yn_tr[b]])); pair_y.append(0.0)
    for b in [b for b in np.argsort(D_prof[a])[1:15] if D_par[a, b] > 0.8][:3]:
        pair_x.append(np.concatenate([Ztr[a], yn_tr[b]])); pair_y.append(0.0)
pair_x = torch.tensor(np.array(pair_x), dtype=torch.float32)
pair_y = torch.tensor(np.array(pair_y), dtype=torch.float32)

ranker = nn.Sequential(nn.Linear(19, 256), nn.ReLU(), nn.Dropout(0.2),
                       nn.Linear(256, 256), nn.ReLU(), nn.Dropout(0.2),
                       nn.Linear(256, 1))
opt = torch.optim.Adam(ranker.parameters(), lr=1e-3, weight_decay=1e-5)
for _ in range(3000):
    idx = torch.randperm(len(pair_x))[:256]
    loss = bce(ranker(pair_x[idx]).squeeze(-1), pair_y[idx])
    opt.zero_grad()
    loss.backward()
    opt.step()
ranker.eval()

cands_norm = (val_cands - y_mean) / y_std
feat = np.concatenate([np.repeat(Z[idx_val][:, None, :], 25, 1), cands_norm], -1)
with torch.no_grad():
    scores = ranker(torch.tensor(feat, dtype=torch.float32)
                    .reshape(-1, 19)).reshape(len(idx_val), 25).numpy()


def select_diverse(c, s, k=5):
    order = np.argsort(-s)
    chosen = [order[0]]
    for _ in range(k - 1):
        best_j, best_g = None, -np.inf
        for j in order:
            if j in chosen:
                continue
            g = min(np.linalg.norm(c[j] - c[i]) for i in chosen) + 0.02 * s[j]
            if g > best_g:
                best_g, best_j = g, j
        chosen.append(best_j)
    return chosen


top5 = np.stack([val_cands[i][select_diverse(cands_norm[i], scores[i])]
                 for i in range(len(idx_val))])


def oracle5(j, log_scale=False):
    if log_scale:
        return 100 * np.mean(np.abs(top5[:, :, j] - true_log[:, j:j+1]).min(1)
                             / np.abs(true_log[:, j]))
    return 100 * np.mean(np.abs(10**top5[:, :, j] - true_lin[:, j:j+1]).min(1)
                         / true_lin[:, j])


auc = roc_auc_score(y_cls[idx_val], val_proba)
print("параметр | TCN                  | CNN (dl_pro) | публикация")
print("---------+----------------------+--------------+-----------")
print(f"XUVInt   | {oracle5(0):8.1f}%            | 10.8%        | 20.4%")
print(f"Helium   | {oracle5(1):8.1f}%            | 9.7%         | 17.9%")
print(f"logMsw   | {oracle5(2, True):8.1f}%            | 0.6%         | 1.8%")
print(f"H2a AUC  | {auc:8.3f}             | 0.995        | 0.989")

параметр | TCN                  | CNN (dl_pro) | публикация
---------+----------------------+--------------+-----------
XUVInt   |     16.9%            | 10.8%        | 20.4%
Helium   |     11.3%            | 9.7%         | 17.9%
logMsw   |      0.8%            | 0.6%         | 1.8%
H2a AUC  |    0.981             | 0.995        | 0.989


## Выводы

* TCN честно отработал как энкодер последовательности: каузальные
  дилатированные свёртки покрывают весь профиль (рецептивное поле 121 > 101),
  а чтение с последнего таймстепа сохраняет позиционную информацию —
  критичную для `logMsw` (доплеровский сдвиг пика).
* Сравнение с CNN-энкодером из `dl_pro.ipynb` в таблице выше: протокол,
  сплит, головы и ранкер идентичны, отличается только энкодер, поэтому
  разница метрик — это вклад именно архитектуры.
* Особенность адаптации: каузальность для нашей задачи не обязательна
  (профиль — не временной ряд с причинностью), но мы сохранили её для
  верности статье; двунаправленный вариант (симметричный паддинг) —
  естественное направление для улучшения.